In [31]:
import pandas as pd

# ========= 1) Ler a base =========
dados = [
    [1, "infantil", "miopia", "não", "reduzida", "nenhuma"],
    [2, "infantil", "miopia", "sim", "normal", "gelatinosa"],
    [3, "infantil", "hipermetropia", "não", "normal", "gelatinosa"],
    [4, "infantil", "hipermetropia", "sim", "normal", "dura"],
    [5, "adolescente", "miopia", "não", "reduzida", "gelatinosa"],
    [6, "adolescente", "miopia", "sim", "reduzida", "nenhuma"],
    [7, "adolescente", "miopia", "não", "normal", "dura"],
    [8, "adolescente", "hipermetropia", "não", "reduzida", "gelatinosa"],
    [9, "adolescente", "hipermetropia", "sim", "normal", "dura"],
    [10, "adulto", "miopia", "não", "normal", "gelatinosa"],
    [11, "adulto", "miopia", "sim", "normal", "dura"],
    [12, "adulto", "miopia", "sim", "normal", "gelatinosa"],
    [13, "adulto", "hipermetropia", "não", "reduzida", "nenhuma"],
    [14, "adulto", "hipermetropia", "sim", "normal", "gelatinosa"],
    [15, "adulto", "hipermetropia", "não", "normal", "gelatinosa"],
]

df = pd.DataFrame(dados, columns=[
    "#", "Idade", "Diagnóstico", "Astigmatismo", "Taxa lacrimal", "Tipo de lente"
])

# ========= 2) Definir classe e atributos =========
alpha = 1
classe_col = "Tipo de lente"

atributos = [col for col in df.columns if col not in [classe_col, "#"]]
classes = sorted(df[classe_col].unique())

valores_atributos = {
    atributo: sorted(df[atributo].unique())
    for atributo in atributos
}

print("Classes:", classes)
print("Atributos:", atributos)

# ========= 3) Tabela de contagens e probabilidades suavizadas =========
linhas = []

for atributo in atributos:
    k = len(valores_atributos[atributo])

    for classe in classes:
        dados_classe = df[df[classe_col] == classe]
        total_classe = len(dados_classe)

        for valor in valores_atributos[atributo]:
            contagem = len(dados_classe[dados_classe[atributo] == valor])
            prob = (contagem + alpha) / (total_classe + alpha * k)

            linhas.append([
                atributo,
                valor,
                classe,
                contagem,
                total_classe,
                k,
                prob
            ])

tabela_suavizada = pd.DataFrame(
    linhas,
    columns=[
        "Atributo",
        "Valor",
        "Classe",
        "Contagem",
        "Total Classe",
        "k",
        "Probabilidade Suavizada"
    ]
)

print("\nTabela de contagens e probabilidades suavizadas:")
display(tabela_suavizada)

# ========= 4) Paciente =========
paciente = {
    "Idade": "infantil",
    "Diagnóstico": "hipermetropia",
    "Astigmatismo": "não",
    "Taxa lacrimal": "reduzida"
}

# ========= 5) Naive Bayes para esse paciente =========
probs = {}

for classe in classes:
    dados_classe = df[df[classe_col] == classe]
    total_classe = len(dados_classe)

    # probabilidade a priori
    prob = total_classe / len(df)

    print(f"\nClasse: {classe}")
    print(f"Prior = {total_classe}/{len(df)} = {prob:.6f}")

    for atributo, valor in paciente.items():
        k = len(valores_atributos[atributo])
        contagem = len(dados_classe[dados_classe[atributo] == valor])
        p_cond = (contagem + alpha) / (total_classe + alpha * k)

        print(
            f"P({atributo}={valor} | {classe}) = "
            f"({contagem}+{alpha}) / ({total_classe}+{alpha}*{k}) = {p_cond:.6f}"
        )

        prob *= p_cond

    probs[classe] = prob
    print(f"Probabilidade não normalizada = {prob:.8f}")

# ========= 6) Normalização =========
soma = sum(probs.values())
probs_normalizadas = {classe: prob / soma for classe, prob in probs.items()}

print("\nProbabilidades normalizadas:")
for classe, prob in probs_normalizadas.items():
    print(f"{classe}: {prob:.6f}")

previsao = max(probs_normalizadas, key=probs_normalizadas.get)
print(f"\nPrevisão final: {previsao}")

Classes: ['dura', 'gelatinosa', 'nenhuma']
Atributos: ['Idade', 'Diagnóstico', 'Astigmatismo', 'Taxa lacrimal']

Tabela de contagens e probabilidades suavizadas:


,Atributo,Valor,Classe,Contagem,Total Classe,k,Probabilidade Suavizada
0,Idade,adolescente,dura,2,4,3,0.428571
1,Idade,adulto,dura,1,4,3,0.285714
2,Idade,infantil,dura,1,4,3,0.285714
3,Idade,adolescente,gelatinosa,2,8,3,0.272727
4,Idade,adulto,gelatinosa,4,8,3,0.454545
5,Idade,infantil,gelatinosa,2,8,3,0.272727
6,Idade,adolescente,nenhuma,1,3,3,0.333333
7,Idade,adulto,nenhuma,1,3,3,0.333333
8,Idade,infantil,nenhuma,1,3,3,0.333333
9,Diagnóstico,hipermetropia,dura,2,4,2,0.500000



Classe: dura
Prior = 4/15 = 0.266667
P(Idade=infantil | dura) = (1+1) / (4+1*3) = 0.285714
P(Diagnóstico=hipermetropia | dura) = (2+1) / (4+1*2) = 0.500000
P(Astigmatismo=não | dura) = (1+1) / (4+1*2) = 0.333333
P(Taxa lacrimal=reduzida | dura) = (0+1) / (4+1*2) = 0.166667
Probabilidade não normalizada = 0.00211640

Classe: gelatinosa
Prior = 8/15 = 0.533333
P(Idade=infantil | gelatinosa) = (2+1) / (8+1*3) = 0.272727
P(Diagnóstico=hipermetropia | gelatinosa) = (4+1) / (8+1*2) = 0.500000
P(Astigmatismo=não | gelatinosa) = (5+1) / (8+1*2) = 0.600000
P(Taxa lacrimal=reduzida | gelatinosa) = (2+1) / (8+1*2) = 0.300000
Probabilidade não normalizada = 0.01309091

Classe: nenhuma
Prior = 3/15 = 0.200000
P(Idade=infantil | nenhuma) = (1+1) / (3+1*3) = 0.333333
P(Diagnóstico=hipermetropia | nenhuma) = (1+1) / (3+1*2) = 0.400000
P(Astigmatismo=não | nenhuma) = (2+1) / (3+1*2) = 0.600000
P(Taxa lacrimal=reduzida | nenhuma) = (3+1) / (3+1*2) = 0.800000
Probabilidade não normalizada = 0.01280000



In [32]:
import pandas as pd

# base com 15 linhas
dados = [
    [1, "infantil", "miopia", "não", "reduzida", "nenhuma"],
    [2, "infantil", "miopia", "sim", "normal", "gelatinosa"],
    [3, "infantil", "hipermetropia", "não", "normal", "gelatinosa"],
    [4, "infantil", "hipermetropia", "sim", "normal", "dura"],
    [5, "adolescente", "miopia", "não", "reduzida", "gelatinosa"],
    [6, "adolescente", "miopia", "sim", "reduzida", "nenhuma"],
    [7, "adolescente", "miopia", "não", "normal", "dura"],
    [8, "adolescente", "hipermetropia", "não", "reduzida", "gelatinosa"],
    [9, "adolescente", "hipermetropia", "sim", "normal", "dura"],
    [10, "adulto", "miopia", "não", "normal", "gelatinosa"],
    [11, "adulto", "miopia", "sim", "normal", "dura"],
    [12, "adulto", "miopia", "sim", "normal", "gelatinosa"],
    [13, "adulto", "hipermetropia", "não", "reduzida", "nenhuma"],
    [14, "adulto", "hipermetropia", "sim", "normal", "gelatinosa"],
    [15, "adulto", "hipermetropia", "não", "normal", "gelatinosa"],
]

df = pd.DataFrame(dados, columns=[
    "#", "Idade", "Diagnóstico", "Astigmatismo", "Taxa lacrimal", "Tipo de lente"
])

alpha = 1

classe_col = "Tipo de lente"
classes = sorted(df[classe_col].unique())

idades = sorted(df["Idade"].unique())
diagnosticos = sorted(df["Diagnóstico"].unique())
astigs = sorted(df["Astigmatismo"].unique())
taxas = sorted(df["Taxa lacrimal"].unique())

# -------- Prior P(C) --------
prior_rows = []
for c in classes:
    cont = (df[classe_col] == c).sum()
    prob = (cont + alpha) / (len(df) + alpha * len(classes))
    prior_rows.append([c, cont, prob])

t_prior = pd.DataFrame(prior_rows, columns=["Classe", "Contagem", "Probabilidade"])

# -------- P(Idade | Classe) --------
rows_idade = []
for c in classes:
    sub = df[df[classe_col] == c]
    total = len(sub)
    k = len(idades)
    for idade in idades:
        cont = (sub["Idade"] == idade).sum()
        prob = (cont + alpha) / (total + alpha * k)
        rows_idade.append([c, idade, cont, prob])

t_idade = pd.DataFrame(rows_idade, columns=["Classe", "Idade", "Contagem", "Probabilidade"])

# -------- P(Diagnóstico | Classe) --------
rows_diag = []
for c in classes:
    sub = df[df[classe_col] == c]
    total = len(sub)
    k = len(diagnosticos)
    for d in diagnosticos:
        cont = (sub["Diagnóstico"] == d).sum()
        prob = (cont + alpha) / (total + alpha * k)
        rows_diag.append([c, d, cont, prob])

t_diag = pd.DataFrame(rows_diag, columns=["Classe", "Diagnóstico", "Contagem", "Probabilidade"])

# -------- P(Astigmatismo | Classe) --------
rows_ast = []
for c in classes:
    sub = df[df[classe_col] == c]
    total = len(sub)
    k = len(astigs)
    for a in astigs:
        cont = (sub["Astigmatismo"] == a).sum()
        prob = (cont + alpha) / (total + alpha * k)
        rows_ast.append([c, a, cont, prob])

t_ast = pd.DataFrame(rows_ast, columns=["Classe", "Astigmatismo", "Contagem", "Probabilidade"])

# -------- P(Taxa | Idade, Classe) --------
rows_taxa = []
for c in classes:
    for idade in idades:
        sub = df[(df[classe_col] == c) & (df["Idade"] == idade)]
        total = len(sub)
        k = len(taxas)
        for taxa in taxas:
            cont = (sub["Taxa lacrimal"] == taxa).sum()
            prob = (cont + alpha) / (total + alpha * k)
            rows_taxa.append([c, idade, taxa, cont, prob])

t_taxa = pd.DataFrame(rows_taxa, columns=[
    "Classe", "Idade", "Taxa lacrimal", "Contagem", "Probabilidade"
])

print("P(C)")
display(t_prior)

print("P(Idade | Classe)")
display(t_idade)

print("P(Diagnóstico | Classe)")
display(t_diag)

print("P(Astigmatismo | Classe)")
display(t_ast)

print("P(Taxa lacrimal | Idade, Classe)")
display(t_taxa)

# -------- Reclassificação do paciente --------
paciente = {
    "Idade": "infantil",
    "Diagnóstico": "hipermetropia",
    "Astigmatismo": "não",
    "Taxa lacrimal": "reduzida"
}

scores = {}

for c in classes:
    sub_c = df[df[classe_col] == c]

    p_c = ((df[classe_col] == c).sum() + alpha) / (len(df) + alpha * len(classes))
    p_i = ((sub_c["Idade"] == paciente["Idade"]).sum() + alpha) / (len(sub_c) + alpha * len(idades))
    p_d = ((sub_c["Diagnóstico"] == paciente["Diagnóstico"]).sum() + alpha) / (len(sub_c) + alpha * len(diagnosticos))
    p_a = ((sub_c["Astigmatismo"] == paciente["Astigmatismo"]).sum() + alpha) / (len(sub_c) + alpha * len(astigs))

    sub_ci = df[(df[classe_col] == c) & (df["Idade"] == paciente["Idade"])]
    p_t = ((sub_ci["Taxa lacrimal"] == paciente["Taxa lacrimal"]).sum() + alpha) / (len(sub_ci) + alpha * len(taxas))

    score = p_c * p_i * p_d * p_a * p_t
    scores[c] = score

soma = sum(scores.values())
probs_norm = {c: scores[c] / soma for c in scores}

print("\nScores não normalizados:")
for c, s in scores.items():
    print(f"{c}: {s:.8f}")

print("\nProbabilidades normalizadas:")
for c, p in probs_norm.items():
    print(f"{c}: {p:.6f}")

print("\nPrevisão final:", max(probs_norm, key=probs_norm.get))

P(C)


,Classe,Contagem,Probabilidade
0,dura,4,0.277778
1,gelatinosa,8,0.500000
2,nenhuma,3,0.222222


P(Idade | Classe)


,Classe,Idade,Contagem,Probabilidade
0,dura,adolescente,2,0.428571
1,dura,adulto,1,0.285714
2,dura,infantil,1,0.285714
3,gelatinosa,adolescente,2,0.272727
4,gelatinosa,adulto,4,0.454545
5,gelatinosa,infantil,2,0.272727
6,nenhuma,adolescente,1,0.333333
7,nenhuma,adulto,1,0.333333
8,nenhuma,infantil,1,0.333333


P(Diagnóstico | Classe)


...

P(Astigmatismo | Classe)


,Classe,Astigmatismo,Contagem,Probabilidade
0,dura,não,1,0.333333
1,dura,sim,3,0.666667
2,gelatinosa,não,5,0.600000
3,gelatinosa,sim,3,0.400000
4,nenhuma,não,2,0.600000
5,nenhuma,sim,1,0.400000


P(Taxa lacrimal | Idade, Classe)


,Classe,Idade,Taxa lacrimal,Contagem,Probabilidade
0,dura,adolescente,normal,2,0.750000
1,dura,adolescente,reduzida,0,0.250000
2,dura,adulto,normal,1,0.666667
3,dura,adulto,reduzida,0,0.333333
4,dura,infantil,normal,1,0.666667
5,dura,infantil,reduzida,0,0.333333
6,gelatinosa,adolescente,normal,0,0.250000
7,gelatinosa,adolescente,reduzida,2,0.750000
8,gelatinosa,adulto,normal,4,0.833333
9,gelatinosa,adulto,reduzida,0,0.166667



Scores não normalizados:
dura: 0.00440917
gelatinosa: 0.01022727
nenhuma: 0.01185185

Probabilidades normalizadas:
dura: 0.166457
gelatinosa: 0.386105
nenhuma: 0.447437

Previsão final: nenhuma
